# Scene3D

## I. Getting Started Guide

### 1. Scene3D Model Introduction

Scene3D is a monocular depth estimation network within the VisionPilot suite designed to infer per-pixel scene depth from a single
front-view image. It follows an Encoder-Context-Decoder pattern, combining pretrained semantic features with a dedicated depth-context
module to recover geometry-aware structure.

The architecture components include:

- A pretrained Backbone (wrapped by `PreTrainedBackbone`) that extracts hierarchical multi-scale features from the input image. 
In this upstream stage, backbone parameters are frozen to preserve robust pretrained representations.

- A DepthContext module that consumes the deepest feature map, performs global spatial pooling, passes it through MLP layers, reshapes
the compact representation into a 2D context map, and expands it with convolutional layers.

- A residual attention-style fusion in `DepthContext` that injects context into deep features using $Context \times Features + Features$,
helping the model encode global scene layout while preserving local feature detail.

- A Scene3D Neck (decoder) with progressive transposed-convolution upsampling and skip connections from encoder stages (`features[3]`,
`features[2]`, `features[1]`) to recover spatial resolution and fine structural cues.

- A Scene3D Head that performs the final upsampling with an additional skip connection from `features[0]`, followed by convolutional
prediction layers that produce a single-channel dense depth output.

As a scene geometry model, Scene3D processes raw perspective frames and predicts dense depth for each pixel in real time. This allows
downstream driving stacks to estimate free space and object distance cues without requiring LiDAR, while remaining lightweight enough for
embedded deployment workflows.

![](../../Media/Scene3D_GIF_2.gif)

### 2. Environment Setup

**If you have done these steps of environment setup in other End-to-End Model Tutorials, please skip this and move forward to the next subsection `3. Download Models`.**

First, we need to clone the repository to your local environment. We will use the official repository from the
Autoware Foundation.

In [ ]:
import os

base_dir = os.getcwd()

# Clone Autoware VisionPilot repo, if not yet done
if not os.path.exists("./autoware_vision_pilot"):
    !git clone https://github.com/autowarefoundation/autoware_vision_pilot.git

The VisionPilot project relies on several key libraries, including **PyTorch** for model execution and
**OpenCV** for image processing.

We will use `pip` to install the required dependencies directly from the `requirements.txt` file provided in 
the `Models` directory.

In [ ]:
# Install required dependencies
%pip install --upgrade pip
%pip install -r autoware_vision_pilot/Models/requirements.txt

# Verify the installation (checking torch as a primary dependency)
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA availability: {torch.cuda.is_available()}")

### 3. Download Models

You can manually download the model weight files, using the links below. Subsequent tutorial sections assume
you already downloaded them and put them inside directory: 
`autoware_vision_pilot/Tutorials/E2E_Models/weights/`.

- [Link to Download Pytorch Model Weights (*.pth)](https://drive.google.com/file/d/1MrKhfEkR0fVJt-SdZEc0QwjwVDumPf7B/view?usp=sharing)
- [Link to Download Traced Pytorch Model (*.pt)](https://drive.google.com/file/d/1-LO3j2YCvwxeNLzyLrnzEwalTrYUZgK0/view?usp=drive_link)
- [Link to Download ONNX FP32 Weights (*.onnx)](https://drive.google.com/file/d/19gMPt_1z4eujo4jm5XKuH-8eafh-wJC6/view?usp=drive_link)

You can also use below script block, which uses `gdown` to download above model weights into such default directory.

In [ ]:
%pip install gdown

import gdown

model_dir = "./weights"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

metadata = {
    "pth"   : {
        "url"       : "https://drive.google.com/uc?id=1MrKhfEkR0fVJt-SdZEc0QwjwVDumPf7B",
        "filepath"  : os.path.join(base_dir, model_dir, "scene3d.pth")
    },
    "pt"    : {
        "url"       : "https://drive.google.com/uc?id=1-LO3j2YCvwxeNLzyLrnzEwalTrYUZgK0",
        "filepath"  : os.path.join(base_dir, model_dir, "scene3d.pt")
    },
    "onnx"  : {
        "url"       : "https://drive.google.com/uc?id=19gMPt_1z4eujo4jm5XKuH-8eafh-wJC6",
        "filepath"  : os.path.join(base_dir, model_dir, "scene3d.onnx")
    }
}

for weight_type in metadata:

    url         = metadata[weight_type]["url"]
    filepath    = metadata[weight_type]["filepath"]

    if not os.path.exists(filepath):
        print(f"Downloading Scene3D {weight_type} weights...")
        gdown.download(url, filepath, quiet = False)
    else:
        print(f"Scene3D {weight_type} weights already exist at {filepath}. Skipping download.")

## II. Quick Test/Inference

Pre-requisites: please ensure that you have completed the above **Environment Setup** and **Download Models** first.

### 1. Image Inference

The `image_visualization.py` script facilitates batch processing of images by loading a pre-trained model checkpoint, performing a forward pass on each image, and overlaying colorized depth predictions onto the original frames.

#### a. Run Batch Image Inference

To run this in any environment, we must navigate to the specific visualization directory to ensure the script's internal import paths resolve correctly.

Key Script Parameters:
- `-p / --model_checkpoint_path`: location of your `.pth` file containing the trained weights.
- `-i / --input_image_dirpath`: directory containing `.png`, `.jpg`, or `.jpeg` files to process.
- `-o / --output_image_dirpath`: destination directory where the script saves visualization files as `[image_id]_result.png`.

In [ ]:
# 1. Path declaration
# Here we use the previously downloaded Pytorch weight file.
# For input and output directories, you can change them to your preferred paths.
# For now we use default sample image assets in the repository.
CHECKPOINT = metadata["pth"]["filepath"]
INPUT_DIR = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/images/"
)
OUTPUT_DIR = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/scene3d/images/"
)

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/Scene3D && \
python3 image_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_DIR)} \
    -o {os.path.abspath(OUTPUT_DIR)}

#### b. Display Results in Notebook

After running the inference, we can use matplotlib and PIL to visualize the generated depth overlays directly within this notebook.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Path to the newly created results
result_files = sorted([
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".png")
])

if (result_files):
    # Display the first 3 results
    fig, axes = plt.subplots(
        1, 3,
        figsize = (20, 10)
    )
    for i, ax in enumerate(axes):
        if (i < len(result_files)):
            img_path = os.path.join(OUTPUT_DIR, result_files[i])
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"Result: {result_files[i]}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No result images found. Check your output directory.")

### 2. Video Inference

In this subsection, we will apply the Scene3D model to video streams.
The `video_visualization.py` script processes video files frame-by-frame, performs depth inference, and compiles the results into a new annotated video file.

The video inference pipeline utilizes `OpenCV` for frame extraction/writing and uses a viridis colormap overlay for visualizing relative depth intensity.

Key script parameters:

- `-p / --model_checkpoint_path`: path to the trained Scene3D weights.
- `-i / --input_video_filepath`: path to the source video file.
- `-o / --output_video_filepath`: destination output path for the visualized video.
- `-v / --vis`: optional flag to enable a pop-up window showing frames as they are processed.

Technical details:

- The script resizes each frame to $(640, 320)$ before inference, then resizes visualization outputs to $(1280, 720)$ for the final video.
- It applies a viridis colormap and alpha blending with $\alpha = 0.97$ to produce a depth-overlay visualization.
- Video capture and writer pointers are released automatically upon completion or stream end.

In [ ]:
# 1. Path declaration
# Similarly as image inference above:
# - Pytorch weight file.
# - Input/Output paths can be changed to preferred.
# - Using defaults for now.
INPUT_VIDEO_PATH = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/videos/highway_normal.mp4"
)
OUTPUT_VIDEO_PATH = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/scene3d/videos/highway_normal_output"
)

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/Scene3D && \
python3 video_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_VIDEO_PATH)} \
    -o {os.path.abspath(OUTPUT_VIDEO_PATH)}

## III. Model Training

### 1. Model Training on New Data

#### a. Load pre-trained model or load vanilla un-trained model for training from scratch

In this repository, model loading behavior is controlled in `Models/training/train_scene_3d.py`, then passed into `Scene3DTrainer(...)`.

- If `--load_from_save` is **not** provided, the script creates `Scene3DTrainer(pretrained_checkpoint_path=...)`. 
In this mode, the upstream `SceneSeg` backbone is initialized from a pre-trained checkpoint, while Scene3D downstream layers are trained from random initialization.
- If `--load_from_save` **is** provided, the script creates `Scene3DTrainer(checkpoint_path=..., is_pretrained=True)`. 
In this mode, the full Scene3D network (including upstream-loaded weights embedded in that checkpoint) is restored and training continues from that saved state.

Use this pattern:

```bash
# A) Train Scene3D with pre-trained upstream SceneSeg weights
python3 Models/training/train_scene_3d.py \
  -m /absolute/path/to/sceneseg_checkpoint.pth \
  -r /absolute/path/to/scene3d_dataset_root/ \
  -s /absolute/path/to/save_models/ \
  -t /absolute/path/to/save_test_images/

# B) Resume / fine-tune from a saved Scene3D checkpoint
python3 Models/training/train_scene_3d.py \
  -l \
  -c /absolute/path/to/scene3d_checkpoint.pth \
  -r /absolute/path/to/scene3d_dataset_root/ \
  -s /absolute/path/to/save_models/ \
  -t /absolute/path/to/save_test_images/
```

Practical recommendation:
- Use mode **A** when starting a fresh Scene3D run with only upstream pretraining available.
- Use mode **B** when you want to continue or fine-tune an existing Scene3D training run.

Note: in the current implementation, a true full-network "all-random" start is not exposed by CLI. 
Non-resume mode expects a valid pre-trained SceneSeg checkpoint through `-m`.

#### b. Prepare the training datasets

Scene3D data preparation in this repository uses two label-generation styles:

1. **Relative-depth pseudo labels**
2. **Metric-depth labels from LiDAR/stereo datasets**

The scripts in `Models/data_parsing/Scene3D/` support metric-depth dataset creation from:
- Argoverse (`Argoverse/process_argoverse.py`)
- KITTI (`KITTI/process_kitti.py`)
- DDAD (`DDAD/process_ddad.py`)
- DrivingStereo (`DrivingStereo/process_driving_stereo.py`)

All parsers produce aligned RGB + depth + validity-mask samples and rely on the same densification utility (`common/lidar_depth_fill.py`), which performs dilation + morphological closing + median filtering on sparse depth maps.

Each parser saves processed outputs in this structure (names vary a lil bit by script, but concept is the same):

```json
<OUTPUT_ROOT>/
  |----image/        # RGB frames
  |----depth/        # dense/interpolated depth maps (.npy)
  |----validity/     # valid-depth masks (.png)
```

##### i) Argoverse

Expected raw content includes disparity maps + rectified stereo images + calibration JSON.

```bash
python3 Models/data_parsing/Scene3D/Argoverse/process_argoverse.py \
  --root /path/to/argoverse_raw/ \
  --save /path/to/processed/ARGOVERSE
```

##### ii) KITTI

Expected raw content includes `train/` depth ground truth and corresponding raw image sequences under `data/`.

```bash
python3 Models/data_parsing/Scene3D/KITTI/process_kitti.py \
  --root /path/to/kitti_raw/ \
  --save /path/to/processed/KITTI/
```

##### iii) DDAD

Requires a valid DDAD dataset JSON used by `SynchronizedSceneDataset`.

```bash
python3 Models/data_parsing/Scene3D/DDAD/process_ddad.py \
  --data /path/to/ddad.json \
  --save /path/to/processed/DDAD
```

##### iv) DrivingStereo

Expected raw content includes `depth-raw/` and `image-raw/`.

```bash
python3 Models/data_parsing/Scene3D/DrivingStereo/process_driving_stereo.py \
  --root /path/to/drivingstereo_raw/ \
  --save /path/to/processed/DRIVINGSTEREO
```

Recommended workflow:

1. Process each dataset into its own output directory first.
2. Spot-check a few samples: image vs depth alignment and validity masks.
3. Merge/curate selected processed outputs into your Scene3D training root used by `train_scene_3d.py` (explained in the next subsection).

Note: parser implementations may include dataset-specific assumptions (index ranges, hard-coded path slicing, crop windows). 
If your local raw layout differs, adjust those constants before full conversion.

#### c. How to load dataset

`train_scene_3d.py` currently loads data through:

- `root + 'Diverse/image/'` (RGB files)
- `root + 'Diverse/relative-depth/'` (depth labels as `.npy`)
- `root + 'Test/'` (images used for periodic visual test outputs during training)

So the expected training root is:

```json
<ROOT_PATH>/
  |----Diverse/
  |    |----image/
  |    |----relative-depth/
  |----Test/
```

How loading works in code:

- `LoadDataScene3D` reads all files from `image/` and all `.npy` files from `relative-depth/`.
- Counts must match exactly (one label per image).
- Train/validation split is automatic: every 20th sample goes to validation, others to training.
- Depth labels are min-max normalized in the loader before being passed to the trainer.

Practical integration from processed metric datasets:

1. Consolidate your curated image files into `<ROOT_PATH>/Diverse/image/`.
2. Consolidate matching depth `.npy` labels into `<ROOT_PATH>/Diverse/relative-depth/`.
3. Keep filenames/indexing aligned so image-label sorting order corresponds one-to-one.
4. Put representative visualization test images in `<ROOT_PATH>/Test/`.

Quick sanity checklist before training:

- Number of files in `Diverse/image/` equals number of files in `Diverse/relative-depth/`.
- All depth labels are valid `.npy` arrays with shape compatible with corresponding images after augmentations.
- `Test/` contains a few valid image files for periodic qualitative outputs.

If you prefer to train directly from metric parser outputs (`depth/` + `validity/`), you can either:

- map/rename `depth/` to `relative-depth/` and use the current loader path convention, or
- modify `train_scene_3d.py`/`load_data_scene_3d.py` paths to point to your custom directory names.

#### d. How to run training

After dataset preparation is complete and paths are configured, training is launched with `train_scene_3d.py`.

##### i) Step 1: Prepare required paths and mode

`train_scene_3d.py` requires these core arguments:
- `-r / --root`: training data root containing `Diverse/` and `Test/` (as described in subsection **c. How to load dataset** above).
- `-s / --model_save_root_path`: directory used when saving `.pth` checkpoints.
- `-t / --test_images_save_root_path`: directory used for periodic test-image predictions.

Then choose one loading mode:
- **Fresh Scene3D training with pre-trained SceneSeg upstream**: provide `-m / --pretrained_checkpoint_path`.
- **Resume from Scene3D checkpoint**: provide `-l` and `-c / --checkpoint_path`.

##### ii) Step 2: Run training

From repository root:

```bash
# A) Fresh Scene3D run (upstream SceneSeg pre-trained)
python3 Models/training/train_scene_3d.py \
  -m /absolute/path/to/sceneseg_checkpoint.pth \
  -r /absolute/path/to/scene3d_root/ \
  -s /absolute/path/to/save_models/ \
  -t /absolute/path/to/save_test_images/

# B) Resume from existing Scene3D checkpoint
python3 Models/training/train_scene_3d.py \
  -l \
  -c /absolute/path/to/scene3d_checkpoint.pth \
  -r /absolute/path/to/scene3d_root/ \
  -s /absolute/path/to/save_models/ \
  -t /absolute/path/to/save_test_images/
```

##### iii) Step 3: What the script does during training

- Builds train/val splits from `LoadDataScene3D` (every 20th sample goes to validation).
- Applies depth augmentations via `Augmentations(..., data_type='DEPTH')`.
- Computes total loss as `mAE_loss + 4 * multi_scale_edge_loss`.
- Uses gradient accumulation to simulate batch updates (`optimizer.step()` every `batch_size` samples).
- Logs training loss every 250 steps and training visualizations every 1000 steps.
- Runs full validation/checkpoint/test-image export every 92,000 global steps.

Current default training settings in script:

- `num_epochs = 5`, loop starts at `epoch = 2` (effective printed epochs are 3 to 5).
- Learning rate is set to `1.25e-5` during loop and batch size is reduced to 3.

#### e. How to visualize training results

This training pipeline provides two main visualization channels:

1. **TensorBoard logs** (training + validation scalar losses, plus periodic train figures)
2. **Saved test predictions** written to your `--test_images_save_root_path` during validation checkpoints

##### a) Open TensorBoard

Run from project root:

```bash
tensorboard --logdir runs --port 6006
```

Then open `http://localhost:6006` in your browser.

What you should see:

- `Train/total_loss`, `Train/mAE_loss`, `Train/edge_loss` logged every 250 steps.
- `Validation/total_loss`, `Validation/mAE_loss`, `Validation/edge_loss` logged when validation is triggered.
- Figure panel `predictions vs. actuals` logged every 1000 training steps via `save_visualization(...)`.

##### b) Inspect saved test predictions and checkpoints

During each validation trigger (`(log_count + 1) % 92000 == 0`), the script:

- Saves model checkpoint files under your `--model_save_root_path`.
- Runs inference on every image in `<ROOT>/Test/` and saves colored depth predictions to `--test_images_save_root_path`.

Saved filenames include iteration/epoch/step metadata, for example:

- `iter_92000_epoch_3_step_1234.pth`
- `iter_92000_epoch_3_step_1234_img_0.png`

Practical note: with small datasets or short runs, the 92,000-step validation interval may not be reached. 
In that case you will still get training TensorBoard logs, but no periodic validation scalars/checkpoints/test-image dumps unless
you lower that interval in `train_scene_3d.py`.

### 2. Building an Inference Pipeline

#### a. Using ROS2 to publish an image, run inference and visualize results

TBD.

#### b. Using IceOryx to publish an image, run inference and visualize results

TBD.

#### c. Using Zenoh to publish an image, run inference and visualize results

TBD.

#### d. Using Gstreamer to get an image, run inference and visualize results

TBD.

## IV. Lite version - Scene3DLite

### 1. Lite version introduction

Scene3DLite is a lightweight monocular **relative depth estimation** model 
designed for efficient deployment on embedded edge devices. It has been 
distilled from large-scale transformer-based depth models (specifically 
`DepthAnythingV2`, as used in the original Scene3D) into a compact 
convolutional architecture while preserving essential depth properties.

#### a. Key characteristics

Scene3DLite maintains:
- Fine depth boundaries around objects.
- Correct near-to-far spatial relationships.
- Coherent depth structure across scenes with scale-consistent predictions.

All of those while dramatically reducing computational overhead!

#### b. Key architecture differences

**Scene3DLite vs. Original Scene3D:**

| Aspect | Scene3D | Scene3DLite |
|--------|----------|--------------|
| **Encoder** | Pre-trained Backbone (varies) | EfficientNet-B1 |
| **Decoder** | Scene3D Neck | Lightweight DeepLabV3+ |
| **Training** | Fine-tuned on metric depth | Distilled from DepthAnythingV2 |
| **Compute (GOPs)** | 224 | 7.78 |
| **FPS (INT8 TensorRT)** | ~10 | 91.4 |
| **Device** | GPU-dominant | Jetson Orin Nano (8GB) |

#### c. Performance metrics (Jetson Orin Nano 8GB)

| Model | GOPs | Backend | Forward [ms] | FPS |
|-------|------|---------|--------------|-----|
| Scene3D | 224 | FP32 ONNX | 168.4 | 5.9 |
| Scene3D | 224 | FP32 TensorRT | 99.7 | 10.0 |
| Scene3DLite | 7.78 | FP32 ONNX | 41.0 | 24.4 |
| Scene3DLite | 7.78 | FP32 TensorRT | 23.4 | 42.7 |
| Scene3DLite | 7.78 | INT8 TensorRT | 10.9 | 91.4 |

The "forward pass" includes Host => Device (H2D) transfer + inference + Device => Host (D2H) transfer.

#### d. When to use Scene3DLite

- Real-time obstacle detection requiring dense depth at 90+ FPS.
- Multi-camera 3D perception where computational budget must be shared across multiple streams.
- Embedded autonomous vehicles with strict power and thermal constraints.
- Free-space estimation for path planning without LiDAR.
- Production deployments requiring INT8-quantized models for maximum efficiency.

### 2. Lite version model weights download

The Scene3DLite model weights are available in both PyTorch and ONNX formats. 
You can manually download them using the links below, or use the automated 
Python script provided in this section.

Pre-trained weights should be placed in the directory: `autoware_vision_pilot/Tutorials/E2E_Models/weights/`.

- [Link to download PyTorch weights (*.pth)](https://drive.google.com/file/d/18YFLGQp0xPmhys-9SFW4BLu91PcOVXlL/view?usp=drive_link).
- [Link to download ONNX FP32 (*.onnx)](https://drive.google.com/file/d/1H3EUps_v_ydTnwxATsfHy_DABGt9ztiT/view?usp=drive_link).

The following script uses `gdown` to download the weights automatically into the default directory.

In [ ]:
import gdown

model_dir = "./weights"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

metadata_lite = {
    "pth"   : {
        "url"       : "https://drive.google.com/uc?id=18YFLGQp0xPmhys-9SFW4BLu91PcOVXlL",
        "filepath"  : os.path.join(base_dir, model_dir, "scene3d_lite.pth")
    },
    "onnx"  : {
        "url"       : "https://drive.google.com/uc?id=1H3EUps_v_ydTnwxATsfHy_DABGt9ztiT",
        "filepath"  : os.path.join(base_dir, model_dir, "scene3d_lite.onnx")
    }
}

for weight_type in metadata_lite:

    url         = metadata_lite[weight_type]["url"]
    filepath    = metadata_lite[weight_type]["filepath"]

    if not os.path.exists(filepath):
        print(f"Downloading Scene3DLite {weight_type.upper()} weights...")
        gdown.download(url, filepath, quiet = False)
    else:
        print(f"Scene3DLite {weight_type.upper()} weights already exist at {filepath}. Skipping download.")

### 3. Image Inference

Scene3DLite supports the same inference workflow as the original Scene3D, with 
identical visualization scripts. The key advantage is the dramatically faster 
inference speed while maintaining high-quality relative depth estimation.

#### a. Run Batch Image Inference

The same visualization scripts work for Scene3DLite. Simply provide the path to the Scene3DLite checkpoint instead.

In [ ]:
# 1. Path declaration for Scene3DLite
CHECKPOINT_LITE = metadata_lite["pth"]["filepath"]
INPUT_DIR_LITE = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/images/"
)
OUTPUT_DIR_LITE = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/scene3d_lite/images/"
)

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/Scene3D && \
python3 image_visualization.py \
    -p {os.path.abspath(CHECKPOINT_LITE)} \
    -i {os.path.abspath(INPUT_DIR_LITE)} \
    -o {os.path.abspath(OUTPUT_DIR_LITE)}

#### b. Display Results in Notebook

In [ ]:
# Display Scene3DLite inference results
result_files_lite = sorted([
    f for f in os.listdir(OUTPUT_DIR_LITE)
    if f.endswith(".png")
])

if (result_files_lite):
    # Display the first 3 results
    fig, axes = plt.subplots(
        1, 3,
        figsize = (20, 10)
    )
    for i, ax in enumerate(axes):
        if (i < len(result_files_lite)):
            img_path = os.path.join(OUTPUT_DIR_LITE, result_files_lite[i])
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"Scene3DLite Result: {result_files_lite[i]}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No result images found. Check your output directory.")

### 4. Video Inference

The Scene3DLite model can be applied to video streams in the same way as the 
original Scene3D. The `video_visualization.py` script processes each frame with 
Scene3DLite's ultra-fast inference engine, producing smooth, real-time depth 
overlays.

Key advantages when processing video with Scene3DLite:
- Process video at 90+ FPS on embedded hardware with INT8 quantization.
- Single-frame depth estimation with ultra low latency, in under 11ms on Jetson platforms.
- Consistent depth quality, with steady edge sharpness and relative depth ordering from the teacher model.
- Optimized for continuous real-time operation on autonomous vehicles.

In [ ]:
# 1. Path declaration for Scene3DLite video inference
INPUT_VIDEO_LITE = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/videos/highway_normal.mp4"
)
OUTPUT_VIDEO_LITE = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/scene3d_lite/videos/highway_normal_output"
)

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/Scene3D && \
python3 video_visualization.py \
    -p {os.path.abspath(CHECKPOINT_LITE)} \
    -i {os.path.abspath(INPUT_VIDEO_LITE)} \
    -o {os.path.abspath(OUTPUT_VIDEO_LITE)}